# FactOwl

## 0. Setting up imports and environment variables

In [1]:
!python --version

Python 3.10.13


In [2]:
!ls /my_dir/factowl

LICENSE    cli.py  example.ipynb  factowl.egg-info  setup.py
README.md  data    factowl	  factowl.yml


In [3]:
# !pip -q install wikipedia

In [4]:
%cd /my_dir/factowl

/my_dir/factowl


/opt/conda/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: using dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [5]:
!git status .

fatal: detected dubious ownership in repository at '/my_dir/factowl'
To add an exception for this directory, call:

	git config --global --add safe.directory /my_dir/factowl


In [6]:
!pwd

/my_dir/factowl


In [7]:
!nvidia-smi

Mon Aug 11 00:26:12 2025       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.247.01             Driver Version: 535.247.01   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  NVIDIA GeForce RTX 3090        Off | 00000000:41:00.0 Off |                  N/A |
|  0%   30C    P0             116W / 370W |      0MiB / 24576MiB |      0%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--

In [8]:
%cd /my_dir/factowl/factowl/
!pip install -e ./

/my_dir/factowl/factowl
Obtaining file:///my_dir/factowl/factowl
  Preparing metadata (setup.py) ... done
  Attempting uninstall: factowl
    Found existing installation: factowl 1.2.1
    Uninstalling factowl-1.2.1:
      Successfully uninstalled factowl-1.2.1
  Running setup.py develop for factowl


In [9]:
import argparse
import json
import logging
import numpy as np
import os
import pandas as pd

from huggingface_hub import login
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams

from factowl import FactScorerSpedUpVLLM as FactScorer
from factowl.io import save_predictions, save_eval_results, load_simple_json, load_json_generations
from vllm import LLM, SamplingParams
import os

/opt/conda/lib/python3.10/site-packages/torch/utils/_pytree.py:185: FutureWarning: optree is installed but the version is too old to support PyTorch Dynamo in C++ pytree. C++ pytree support is disabled. Please consider upgrading optree using `python3 -m pip install --upgrade 'optree>=0.13.0'`.
  warnings.warn(


INFO 08-11 00:26:31 [__init__.py:239] Automatically detected platform cuda.


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [10]:
os.environ["CUDA_VISIBLE_DEVICES"]="0,1"
# os.environ["VLLM_LOG_LEVEL"] = "WARNING"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [11]:
%cd /my_dir/factowl_evaluation/factscore_data

/my_dir/factowl_evaluation/factscore_data


/opt/conda/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: using dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


## 1. Setting up the parameters for generation

In here, `hf_token` is needed to load the model from HuggingFace, `model_name` is the model's name. Point `data_dir` to the directory, where the downloaded wikipedia dump is located. Your data, which will be evaluated, should be placed in `file_path`. Additionally, create a cache directory and point towards it using `cache_dir` variable.

In [12]:
!ls /my_dir/wikipedia_dumps/factscore_data/data

In [13]:
hf_token = 'hf_token'
model_name = 'meta-llama/Meta-Llama-3-8B-Instruct'
cache_dir = './cachedir/'
data_dir = '/my_dir/wikipedia_dumps/'
file_path = '/my_dir/wikipedia_dumps/data/labeled'
cnp = 1 # Number of context pages retrieved from Wikipedia API.
nsp = 5 # Number of relevant passages retrieved to support a single atomic fact.
# Retrieval source: 'db' for local Wikipedia dump or 'wikipedia_api' for API search
context_type = 'wikipedia_api'
# Set to true for biography-specific postprocessing on FactScore data
is_bio = False
# Always set to true for faster inferenc
batched_fact_generation = True


## 2. Initialize VLLM engine for generation

We use VLLM to increase the efficiency of our factchecking engine.

In [14]:
os.environ["CUDA_VISIBLE_DEVICES"]="0,1"
# os.environ["VLLM_LOG_LEVEL"] = "WARNING"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

vllm_model = LLM(
    model=model_name,
    dtype="half",
    trust_remote_code=True,
)

WARNING 08-11 00:26:44 [config.py:2972] Casting torch.bfloat16 to torch.float16.
INFO 08-11 00:27:10 [config.py:717] This model supports multiple tasks: {'embed', 'generate', 'reward', 'classify', 'score'}. Defaulting to 'generate'.
INFO 08-11 00:27:23 [config.py:2003] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 08-11 00:27:25 [core.py:58] Initializing a V1 LLM engine (v0.8.5.post1) with config: model='meta-llama/Meta-Llama-3-8B-Instruct', speculative_config=None, tokenizer='meta-llama/Meta-Llama-3-8B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, tokenizer_revision=None, trust_remote_code=True, dtype=torch.float16, max_seq_len=8192, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto,  device_config=cuda, decoding_config=DecodingConfig(guided_decoding_backend='auto', reasoning_backe

Loading safetensors checkpoint shards:   0% Completed | 0/4 [00:00<?, ?it/s]


INFO 08-11 00:27:39 [loader.py:458] Loading weights took 5.68 seconds
INFO 08-11 00:27:39 [gpu_model_runner.py:1347] Model loading took 14.9596 GiB and 6.675449 seconds
INFO 08-11 00:27:46 [backends.py:420] Using cache directory: /root/.cache/vllm/torch_compile_cache/77f859cb55/rank_0_0 for vLLM's torch.compile
INFO 08-11 00:27:46 [backends.py:430] Dynamo bytecode transform time: 7.39 s
INFO 08-11 00:27:53 [backends.py:118] Directly load the compiled graph(s) for shape None from the cache, took 5.670 s
INFO 08-11 00:27:54 [monitor.py:33] torch.compile takes 7.39 s in total
INFO 08-11 00:27:56 [kv_cache_utils.py:634] GPU KV cache size: 39,360 tokens
INFO 08-11 00:27:56 [kv_cache_utils.py:637] Maximum concurrency for 8,192 tokens per request: 4.80x
INFO 08-11 00:28:24 [gpu_model_runner.py:1686] Graph capturing finished in 27 secs, took 1.59 GiB
INFO 08-11 00:28:24 [core.py:159] init engine (profile, create kv cache, warmup model) took 44.82 seconds
INFO 08-11 00:28:24 [core_client.py:439

## 3. Set up the names of the target files and evaluation setup.

In here, you can set up the names of the files to check and determine, whether you want to use npm for checking or not.

In [15]:
import logging
# logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s',
#                         datefmt='%Y-%m-%d %H:%M:%S', )

DEBUG=False
if DEBUG:
    logging.basicConfig(level=logging.DEBUG)

## 4. Evaluation on FactScore data 

In [ ]:
model_name = 'meta-llama/Meta-Llama-3-8B-Instruct'
eval_dict = {}

gen_name2abstain = {
    "ChatGPT": "generic",
    "InstructGPT": "generic",
    "PerplexityAI": "perplexity_ai"
}
gen_name2setup = {
    "ChatGPT": "retrieval+llama",
    "InstructGPT": "retrieval+llama",
    "PerplexityAI": "retrieval+llama"
}

gen_names = ["ChatGPT", "InstructGPT", "PerplexityAI"]
eval_dict = {}


for j, gn in enumerate(gen_names):
    st = gen_name2setup[gn]
    print(f"Evaluating {gn}")
    print("model_name", model_name)

    json_p = os.path.join(file_path, f"{gn}.jsonl")
    abstain_type = gen_name2abstain[gn]
    atomic_facts_cache_dir = f"./cache/{gn}-{st}-{context_type}-p{cnp}-c{nsp}/"
    print(f"{atomic_facts_cache_dir=}")
    


    fs = FactScorer(model_name=st,
            data_dir=data_dir,
            vllm_model=vllm_model,
            atomic_facts_cache_dir=atomic_facts_cache_dir,
            verifier_max_tokens=16,
            fact_generator_max_tokens=4096,
            dump_every_int=200,
            cache_dir=cache_dir,
            abstain_detection_type="generic",
            is_bio=True,
            retrieval_device="cuda:0",
            context_type=context_type,
            context_num_pages=cnp,
            num_supporting_contexts=nsp,
            batched_fact_generation=True,
            debug=DEBUG)

    topics, generations = load_json_generations(json_p)
    # topics, generations = topics[:10], generations[:10]

    out = fs.get_score(topics, generations, gamma=10, knowledge_source="enwiki-20230401", verbose=True)
    eval_dict[gn] = out

    p = f"/my_dir/evaluation/eval_results/vllm_demo_hu/sped_up_predictions/pred_{gn}-{st}_eval-vllm_Llama3-inst-{context_type}-p{cnp}-c{nsp}.tsv"
    save_predictions(eval_dict[gn], p, print_res=False)

    p = f"/my_dir/evaluation/eval_results/vllm_demo_hu/sped_up_scores/eval_res_{gn}-{st}_eval-vllm_Llama3-inst-{context_type}-p{cnp}-c{nsp}.tsv"
    save_eval_results(eval_dict[gn], p)

    print(f'Score: {out["score"]}\nRespond ratio: {out["respond_ratio"]}')


[2025-08-11 00:28:24] INFO factscorer_sped_up_vllm.py:50: FactScore is using context retrieval type: wikipedia_api


Evaluating ChatGPT
model_name meta-llama/Meta-Llama-3-8B-Instruct
atomic_facts_cache_dir='./cache/ChatGPT-retrieval+llama-wikipedia_api-p1-c5/'


  0%|          | 0/183 [00:00<?, ?it/s]

Starting fact generation


Processed prompts:   0%|          | 0/434 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

...

  0%|          | 0/183 [00:00<?, ?it/s]

Starting fact generation


Processed prompts:   0%|          | 0/521 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Fact generation took 70.44201421737671 seconds


 32%|███▏      | 59/183 [04:46<12:09,  5.88s/it]

# Evaluation on your data

In [ ]:

base_dir = "./"
atomic_facts_cache_dir = os.path.join(base_dir, "cache/")
save_dir = os.path.join(base_dir, "results/")

predictions_path = os.path.join(save_dir, "predictions.tsv")
scores_path = os.path.join(save_dir, "scores.tsv")

# Load Free-form generations to be evaluated and generation topics
topics, generations = <LOAD_YOUR_DATA_HERE>

fs = FactScorer(model_name=st,
        data_dir=data_dir,
        vllm_model=vllm_model,
        atomic_facts_cache_dir=atomic_facts_cache_dir,
        verifier_max_tokens=16,
        fact_generator_max_tokens=4096,
        dump_every_int=200,
        cache_dir=cache_dir,
        abstain_detection_type="generic",
        is_bio=True,
        retrieval_device="cuda:0",
        context_type=context_type,
        context_num_pages=cnp,
        num_supporting_contexts=nsp,
        batched_fact_generation=True,
        debug=DEBUG)

topics, generations = load_json_generations(json_p)
out = fs.get_score(topics, generations, gamma=10, knowledge_source="enwiki-20230401", verbose=True)

save_predictions(out, predictions_path, print_res=False)
save_eval_results(out, scores_path)

print(f'Score: {out["score"]}\nRespond ratio: {out["respond_ratio"]}')
